# JR-PLS and T-PLS

**JR-PLS**: Joint R-PLS links multiple material matrices `Xi` (one per material type) and recipe matrices `Ri` to a common quality `Y`. **T-PLS** adds a process block `Z` on top of JR-PLS.

Data loading uses `parse_materials` and `reconcile_rows_to_columns` / `reconcile_rows` to align observations.

In [1]:
import pandas as pd
import numpy as np
import pyphi.calc as phi
import pyphi.plots as pp
from bokeh.io import output_notebook
output_notebook(hide_banner=True)
import pyphi.plots as _ppmod
_ppmod.output_file = lambda *a, **kw: None


Will be using the NEOS server in the absence of IPOPT and GAMS


## Load and Reconcile Data

In [2]:
jr, materials = phi.parse_materials('../data/jrpls_tpls_dataset.xlsx', 'Materials')
x = [pd.read_excel('../data/jrpls_tpls_dataset.xlsx', sheet_name=m) for m in materials]
xc, jrc = phi.reconcile_rows_to_columns(x, jr)

quality = pd.read_excel('../data/jrpls_tpls_dataset.xlsx', sheet_name='QUALITY')
process = pd.read_excel('../data/jrpls_tpls_dataset.xlsx', sheet_name='PROCESS')

jrc.append(process)
jrc.append(quality)
AUX     = phi.reconcile_rows(jrc)
JR_     = AUX[:-2]
process = AUX[-2]
quality = AUX[-1]

Ri = {m: j for m, j in zip(materials, JR_)}
Xi = {m: x_ for m, x_ in zip(materials, xc)}

print('Materials:', materials)
print('Quality shape:', quality.shape)


Lot :L001 ratio/qty adds to 5.0
Lot :L002 ratio/qty adds to 5.0
Lot :L003 ratio/qty adds to 5.0
Lot :L004 ratio/qty adds to 5.0
Lot :L005 ratio/qty adds to 5.0
Lot :L006 ratio/qty adds to 5.0
Lot :L007 ratio/qty adds to 5.0
Lot :L008 ratio/qty adds to 5.0
Lot :L009 ratio/qty adds to 5.0
Lot :L010 ratio/qty adds to 5.0
Lot :L011 ratio/qty adds to 5.0
Lot :L012 ratio/qty adds to 5.0
Lot :L013 ratio/qty adds to 5.0
Lot :L014 ratio/qty adds to 5.0
Lot :L015 ratio/qty adds to 5.0
Lot :L016 ratio/qty adds to 5.0
Lot :L017 ratio/qty adds to 5.0
Lot :L018 ratio/qty adds to 5.0
Lot :L019 ratio/qty adds to 5.0
Lot :L020 ratio/qty adds to 5.0
Lot :L021 ratio/qty adds to 5.0
Lot :L022 ratio/qty adds to 5.0
Lot :L023 ratio/qty adds to 5.0
Lot :L024 ratio/qty adds to 5.0
Lot :L025 ratio/qty adds to 5.0
Lot :L026 ratio/qty adds to 5.0
Lot :L027 ratio/qty adds to 5.0
Lot :L028 ratio/qty adds to 5.0
Lot :L029 ratio/qty adds to 5.0
Lot :L030 ratio/qty adds to 5.0
Lot :L031 ratio/qty adds to 5.0
Lot :L03

Materials: ['MAT1', 'MAT2', 'MAT3', 'MAT4', 'MAT5']
Quality shape: (105, 7)


## Build JR-PLS Model

In [3]:
jrpls_obj = phi.jrpls(Xi, Ri, quality, 4)
print('Keys:', list(jrpls_obj.keys()))


phi.jrpls using NIPALS executed on: 2026-03-27 00:08:31.072013
# Iterations for LV #1:  9
# Iterations for LV #2:  29
# Iterations for LV #3:  14
# Iterations for LV #4:  27
--------------------------------------------------------------
LV #     Eig       R2X       sum(R2X)   R2R       sum(R2R)   R2Y       sum(R2Y)
LV #1:    0.005    0.373     0.373      0.042     0.042      0.154     0.154
LV #2:    0.002    0.222     0.595      0.027     0.069      0.024     0.178
LV #3:    0.003    0.191     0.786      0.022     0.091      0.011     0.190
LV #4:    0.002    0.099     0.886      0.017     0.108      0.005     0.194
--------------------------------------------------------------
Keys: ['T', 'P', 'Q', 'U', 'S', 'H', 'V', 'Rscores', 'r2xi', 'r2xpvi', 'r2xpv', 'mx', 'sx', 'r2y', 'r2ypv', 'my', 'sy', 'r2ri', 'r2rpvi', 'mr', 'sr', 'Xhat', 'materials', 'var_t', 'obsidXi', 'varidXi', 'varidX', 'obsidRi', 'varidRi', 'obsidY', 'varidY', 'T2', 'T2_lim99', 'T2_lim95', 'speX', 'speX_lim99', 'speX_

## JR-PLS Plots

In [4]:
pp.r2pv(jrpls_obj, material='MAT4', addtitle='For MAT4')
pp.score_scatter(jrpls_obj, [1, 2])
pp.score_scatter(jrpls_obj, [1, 2], rscores=True)
pp.loadings(jrpls_obj, material='MAT4')
pp.vip(jrpls_obj, plotwidth=1000, material='MAT4', addtitle='MAT4')


## JR-PLS Prediction

New blends are specified as a dict of `{material: [(lot_id, fraction), ...]}`

In [5]:
rnew = {
    'MAT1': [('A0129', 0.558), ('A0130', 0.442)],
    'MAT2': [('Lac0003', 1)],
    'MAT3': [('TLC018', 1)],
    'MAT4': [('M0012', 1)],
    'MAT5': [('CS0017', 1)],
}
preds = phi.jrpls_pred(rnew, jrpls_obj)
print('JR-PLS prediction:', preds)


JR-PLS prediction: {'Tnew': array([ 0.00467448,  0.03591181,  0.08235519, -0.0388187 ]), 'Yhat': array([[32.66981665,  3.40818739, 58.05202629,  3.24189905, 80.22253995,
         2.73302346]]), 'speR': [np.float64(163.5743090639282), np.float64(46.256381785765164), np.float64(28.724139742628594), np.float64(29.086253146411682), np.float64(28.81968899516607)]}


## Build T-PLS Model

T-PLS adds a process (Z) block on top of the JR-PLS material structure.

In [6]:
tpls_obj = phi.tpls(Xi, Ri, process, quality, 4)
pp.r2pv(tpls_obj)
pp.r2pv(tpls_obj, zspace=True)
pp.loadings(tpls_obj, zspace=True)
pp.weighted_loadings(tpls_obj, zspace=True)
pp.vip(tpls_obj, plotwidth=1000, zspace=True, addtitle='Z Space')


phi.tpls using NIPALS executed on: 2026-03-27 00:08:31.734721
# Iterations for LV #1:  6
# Iterations for LV #2:  8
# Iterations for LV #3:  26
# Iterations for LV #4:  34
--------------------------------------------------------------
LV #     R2X       sum(R2X)   R2R       sum(R2R)   R2Z       sum(R2Z)   R2Y       sum(R2Y)
LV #1:   0.373     0.373      0.050     0.050      0.258     0.258      0.198     0.198
LV #2:   0.208     0.580      0.035     0.085      0.153     0.411      0.042     0.240
LV #3:   0.167     0.747      0.015     0.099      0.079     0.490      0.019     0.259
LV #4:   0.117     0.865      0.017     0.117      0.102     0.592      0.007     0.266
--------------------------------------------------------------


## T-PLS Prediction

In [7]:
znew = process[process.iloc[:, 0] == 'L001'].values.reshape(-1)[1:].astype(float)
preds_t = phi.tpls_pred(rnew, znew, tpls_obj)
print('T-PLS prediction:', preds_t)


T-PLS prediction: {'Tnew': array([-0.05797414, -1.70779194,  0.54206053, -0.59333818]), 'Yhat': array([[32.77691325,  3.09117598, 58.566599  ,  3.28075367, 79.89250069,
         2.6464102 ]]), 'speR': [np.float64(167.35121621926297), np.float64(47.94252872129938), np.float64(31.287551536179937), np.float64(31.53269998842489), np.float64(30.848470952231285)], 'speZ': np.float64(3.4571146009975515)}
